# Neural Network + Clustering: An Alternative to Common Methods?

The first step is to scale down the dataset. We do this because we want to make features comparable so the model doesn't get biased by magnitude differences. For example, we have values (like sbytes) that are in the millions, whereas duration is in a few seconds. For k-means clustering (without scaling) the model will think that sbytes is wayyyy more important than duration, which may or may not be true. Distance calculations (like in k means etc) are dominated by large numbers...

So we need to scale to ensure each feature contributes fairly to learning

Steps to take:

1. Split features
   - numerical
   - categorical

2. Numerical pipeline:
   - log transform (if skewed)
   - StandardScaler

3. Categorical pipeline:
   - OneHotEncoder
   - (NO scaling)

4. Combine features

5. Feed into neural network

6. Extract learned features

7. Run k-means

Each notebook runs its own kernel so no variable declarations here will mess up declarations in the main file. Later on when I put this all back into the main file, we will need to remove the import stuff since it is already in the main file

## Splitting Features, Scaling / Encoding

In [2]:
import kagglehub
#this is the UNSW_NB15 dataset which is more modern
path = kagglehub.dataset_download("mrwellsdavid/unsw-nb15")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Owner\.cache\kagglehub\datasets\mrwellsdavid\unsw-nb15\versions\1


In [3]:
#inspect files
import os
os.listdir(path)
#we see all of the fils that we could've downloaded on kaggle

['NUSW-NB15_features.csv',
 'UNSW-NB15_1.csv',
 'UNSW-NB15_2.csv',
 'UNSW-NB15_3.csv',
 'UNSW-NB15_4.csv',
 'UNSW-NB15_LIST_EVENTS.csv',
 'UNSW_NB15_testing-set.csv',
 'UNSW_NB15_training-set.csv']

In [4]:
#but we really only want the training and testing set
import pandas as pd
train_df = pd.read_csv(os.path.join(path, "UNSW_NB15_training-set.csv"))
test_df = pd.read_csv(os.path.join(path, "UNSW_NB15_testing-set.csv"))

#for OUR project we want to combine both of these because more data = better feature learning, and clustering benefits from more samples

df_og = pd.concat([train_df, test_df], axis = 0)

In [20]:
#we need to split our data into categorical and numerical. We will apply one hot encoding on the categorical, and then we will scale the numerical features.
#note that WE ARE NOT scaling the categorical features. This would ruin their semantic meaning.
#we scale the numerical features because they differ vastly in scale... which is bad

#remember that we are using labels so we can use df_og.copy() to create a dataframe of categorical and a dataframe of numerical

df_cat = df_og[['proto', 'service', 'state']].copy() #we want to make a copy of the og datafram with only the categorical features in columns 1,2,3
df_num = df_og.drop(columns=['proto', 'service', 'state', 'id', 'attack_cat']).copy() #make a copy of the og dataframe with everything except the categorical features.

# #THIS IS ONE HOT ENCODING FOR CATEGORICAL FEATURES:
print("STARTING WITH ONE-HOT ENCODING FOR CATEGORICAL FEATURES...")

from sklearn.preprocessing import OneHotEncoder
import pandas as pd

encoder = OneHotEncoder(sparse_output=False)
# Requires a 2D array, so we use double brackets
encoded_array = encoder.fit_transform(df_cat) #these are our three categorical features that we need to encode

# Convert back to DataFrame with original names
encoded_df_cat = pd.DataFrame(encoded_array, columns=encoder.get_feature_names_out())

#THIS IS SCALING FOR NUMERICAL FEATURES:
#we want to use log transformation to reduce the influence of skewed features and outliers. then we can use standardization to scale the features to have a mean of 0 and a standard deviation of 1.
print("MOVING ON TO SCALING NUMERICAL FEATURES...")

from sklearn.preprocessing import FunctionTransformer
import numpy as np
transformer = FunctionTransformer(np.log1p) #log1p is used to handle zero values in the data, as log(0) is undefined. It computes log(1 + x), which allows us to apply the log transformation without encountering issues with zero values.
df_num_transformed = transformer.transform(df_num) #this applies the log transformation to all the numerical features in our df_num dataframe.

#now we need to take the log transformed data and standardize it using StandardScaler
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaled_df_num_transformed = scaler.fit_transform(df_num_transformed) #this standardizes the log transformed data to have a mean of 0 and a standard deviation of 1.




STARTING WITH ONE-HOT ENCODING FOR CATEGORICAL FEATURES...
MOVING ON TO SCALING NUMERICAL FEATURES...


In [24]:
#check if we have one hot encoded correctly
print(encoded_df_cat.shape)
print(df_cat.nunique())

#based on the output, we now have 157 columns. This indicated we've done it correctly because the og data has 133 unique proto, 13 unique service, and 11 unique state. 
#so each unique value in our df_cat became it's own one-hot column!

#check if we have scaled correctly
print(scaled_df_num_transformed.shape)
print(scaled_df_num_transformed.mean(axis=0)) #should be close to 0 because we standardized the data to have a mean of 0
print(scaled_df_num_transformed.std(axis=0)) #should be close to 1


# the output indicates that we have successfully scaled our numerical features to have a mean of 0 and a standard deviation of 1, which is what we wanted to achieve with standardization.

(257673, 157)
proto      133
service     13
state       11
dtype: int64
(257673, 40)
[ 1.76482344e-17 -1.97660226e-16 -5.64743502e-17  5.64743502e-17
 -1.12948700e-16  9.88301128e-17 -7.34166552e-16 -1.12948700e-16
  0.00000000e+00  1.97660226e-16  8.47115252e-17 -5.64743502e-17
 -3.52964688e-17 -1.34126582e-16  2.04719519e-16 -2.11778813e-17
 -4.23557626e-17  1.97660226e-16 -2.11778813e-16 -1.69423050e-16
 -7.05929377e-18 -7.05929377e-18  1.41185875e-17 -4.07674215e-16
 -8.47115252e-17  1.94130579e-17  4.94150564e-17  3.84731510e-16
  1.69423050e-16  3.10608926e-16  2.82371751e-17  5.64743502e-17
  4.23557626e-17  8.73587604e-17  1.50009993e-17  2.82371751e-17
 -8.47115252e-17  1.48245169e-16 -1.76482344e-17  1.69423050e-16]
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


## Combining Features (to feed into Neural Network)

so now that we have our two separate (approrpiately scaled / encoded) dataframes. We can combine them to feed them to the neural network.

In [25]:
scaled_encoded_combined = np.hstack([encoded_df_cat, scaled_df_num_transformed]) 
#this combines the one hot encoded categorical features and the scaled numerical features into one array that we can use for our neural network. 
#we use hstack to stack them horizontally, meaning we are adding columns to our dataset. The one hot encoded features will be on the left and the scaled numerical features will be on the right. 
#this way we preserve row alignment.

print(scaled_encoded_combined.shape) #this should have the same number of rows as our original dataframe and the number of columns should be the sum of the one hot encoded columns and the scaled numerical columns.

(257673, 197)


the output indicates we have succesfully combined the two. we have the same number of rows as outputted above, and the colums is 197 = 157+40 which is also outputted above

## Neural Network (using combined dataframes)

the combined dataframe is called scaled_encoded_combined

We are going to take a base model and edit it to fit our needs. We will use pyTorch "Feedforward Neural Network" because our dataset is tabular, mixed, not sequential and not spatial. This is the standard model for these types of datasets.

In [26]:
%pip install torch

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   - -------------------------------------- 2.9/114.6 MB 15.2 MB/s eta 0:00:08
   -- ------------------------------------- 6.0/114.6 MB 15.4 MB/s eta 0:00:08
   --- ------------------------------------ 9.2/114.6 MB 14.6 MB/s eta 0:00:08
   ---- ----------------------------------- 12.1/114.6 MB 14.8 MB/s eta 0:00:07
   ----- ---------------------------------- 14.7/114.6 MB 14.0 MB/s eta 0:00:08
   ------ --------------------------------- 17.8/114.6 MB 14.2 MB/s eta 0:00:07
   ------- -------------------------------- 21.0/114.6 MB 14.3 MB/s eta 0:00:07
   -------- ------------------------------- 23.1/114.6 MB 14.3 MB/s eta 0:00:07
   -------- ------------------------------- 24.9/114.6 MB 13.1 MB/s eta 0:00:07
   --------- ------------------------------ 27.8/114.6 MB 13.2 MB/s eta 0:00:07
   ---------- ----------------------------- 30.9/114.6


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


we want to use a neural network to learn a lower-dimensional representation of the data that captures important behavioral patterns. This allows clustering to operate on a more structured feature space.”

In [35]:
import torch
import torch.nn as nn

# Define the network architecture
class FeedForwardNN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(FeedForwardNN, self).__init__()

        #this is the encoder which is in charge of learning a compressed representation of our data. It takes in the input size, which is the number of features we have in our dataset, and outputs a compressed representation of our data that we can use for classification.

        self.encoder = nn.Sequential(nn.Linear(input_size, 128), #this is the first layer of our encoder, it takes in the input size and outputs 128 features
                                     nn.ReLU(), #this is the activation function for the first layer, it introduces non-linearity to the model which allows it to learn more complex patterns in the data

                                     nn.Linear(128,64), #then the second layer of our encoder takes in the 128 features from the first layer and outputs 64 features
                                     nn.ReLU(), #then we apply another activation function to introduce more non-linearity.

                                     nn.Linear(64,32), #this is the bottleneck layer of our encoder, it takes in the 64 features from the second layer and outputs 32 features. This is where we are learning a compressed representation of our data.
                                     nn.ReLU(),
        ) #finally we apply another activation function to the bottleneck layer to introduce more non-linearity.
                                     
        #this is the classifier, which is in charge of classifying our data based on the compressed representation learned by the encoder

        self.classifier = nn.Sequential(nn.Linear(32,64), #this is the first layer of our classifier, it takes in the 32 features from the bottleneck layer and outputs 64 features
                                        nn.ReLU(), #this is the activation function for the first layer of our classifier, it introduces non-linearity to the model which allows it to learn more complex patterns in the data
    
                                        nn.Linear(64,num_classes) #then the second layer of our classifier takes in the 64 features from the first layer and outputs num_classes features, which is the number of classes we have in our dataset. This is where we are learning to classify our data based on the compressed representation learned by the encoder.

        )
    
    def forward(self, x):
        features = self.encoder(x) #first we pass our input through the encoder to get the compressed representation of our data
        out = self.classifier(features) #then we pass the compressed representation through the classifier to get our final output
        return out
    
    def extract_features(self, x):
        return self.encoder(x) #this is a helper function that allows us to extract the compressed features from the encoder, which we can use for clustering later on.

X_tensor = torch.tensor(scaled_encoded_combined, dtype=torch.float32)

# Initialize model
input_size = scaled_encoded_combined.shape[1] #this is the number of features we have in our dataset after one hot encoding and scaling
num_classes = len(df_og['attack_cat'].unique()) #this is the number of unique classes we have in our target variable, which is the attack category
model = FeedForwardNN(input_size, num_classes)
outputs = model(X_tensor)

In [37]:
print("MODEL OUTPUTS:")
print(outputs)

print("PREDICTED CLASSES:")
predicted_classes = torch.argmax(outputs, dim=1)
print(predicted_classes)

probabilities = torch.softmax(outputs, dim=1)
print("PROBABILITIES:")
print(probabilities)

MODEL OUTPUTS:
tensor([[ 0.0470,  0.0253,  0.0569,  ..., -0.0153,  0.1729, -0.1252],
        [ 0.0487,  0.0239,  0.0587,  ..., -0.0121,  0.1722, -0.1245],
        [ 0.0479,  0.0241,  0.0584,  ..., -0.0128,  0.1721, -0.1253],
        ...,
        [ 0.0439,  0.0254,  0.0558,  ..., -0.0148,  0.1649, -0.1233],
        [ 0.0480,  0.0236,  0.0534,  ..., -0.0063,  0.1574, -0.1221],
        [ 0.0482,  0.0237,  0.0534,  ..., -0.0062,  0.1573, -0.1222]],
       grad_fn=<AddmmBackward0>)
PREDICTED CLASSES:
tensor([8, 8, 8,  ..., 8, 8, 8])
PROBABILITIES:
tensor([[0.1050, 0.1027, 0.1060,  ..., 0.0986, 0.1191, 0.0884],
        [0.1050, 0.1024, 0.1060,  ..., 0.0988, 0.1188, 0.0883],
        [0.1050, 0.1025, 0.1061,  ..., 0.0988, 0.1189, 0.0883],
        ...,
        [0.1049, 0.1030, 0.1062,  ..., 0.0989, 0.1184, 0.0888],
        [0.1053, 0.1027, 0.1058,  ..., 0.0997, 0.1175, 0.0888],
        [0.1053, 0.1027, 0.1058,  ..., 0.0997, 0.1174, 0.0888]],
       grad_fn=<SoftmaxBackward0>)


## Taking the Output from Neural Network (FeedForwardNN) and Clustering